# Aggregation Function Distance Metric

For a given case (join + group-by + aggregate), we want to measure:
**If the correct aggregation was X but the model chose Y, how wrong is that?**

We compute a 4×4 matrix over `{min, max, mean, sum}` for each distance metric:
- L1 (Manhattan)
- L2 (Euclidean)
- KL Divergence
- Earth Mover's Distance (Wasserstein-1)
- JS Distance (Jensen-Shannon)
- Hellinger Distance
- Total Variation Distance
- Bhattacharyya Distance

Each entry `[i, j]` = distance between the distribution of agg_i results vs agg_j results,
averaged across all numeric columns being aggregated.

**Case:** `length3_41` — join `test_0.csv` (schools) + `test_1.csv` (students) on `school_name`,
then aggregate per school.

In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance, entropy
from scipy.spatial.distance import jensenshannon
import warnings
warnings.filterwarnings('ignore')

BASE = '/home/asurite.ad.asu.edu/jrtandel/transchema/autopipeline-benchmarks/github-pipelines/length3_41/'

schools = pd.read_csv(BASE + 'test_0.csv', index_col=0)
students = pd.read_csv(BASE + 'test_1.csv', index_col=0)
target   = pd.read_csv(BASE + 'target.csv', index_col=0)

print('Schools:', schools.shape)
print(schools.head(3))
print('\nStudents:', students.shape)
print(students.head(3))
print('\nTarget:', target.shape)
print(target.head(3))

Schools: (15, 5)
   School ID           school_name      type  size   budget
0          0     Huang High School  District  2917  1910635
1          1  Figueroa High School  District  2949  1884411
2          2   Shelton High School   Charter  1761  1056600

Students: (19585, 7)
   Student ID     student_name gender grade        school_name  reading_score  \
0           0     Paul Bradley      M   9th  Huang High School             66   
1           1     Victor Smith      M  12th  Huang High School             94   
2           2  Kevin Rodriguez      M  12th  Huang High School             90   

   math_score  
0          79  
1          61  
2          60  

Target: (8, 4)
   Total Students  Total School Budget  Average Math Score  \
0            1468               917500           83.351499   
1            1761              1056600           83.359455   
2            1858              1081356           83.061895   

   Average Reading Score  
0              83.816757  
1            

In [96]:
# Join and identify numeric columns to aggregate
joined = students.merge(schools, on='school_name', how='left')
print('Joined shape:', joined.shape)
print(joined.head(3))

# Columns we'll aggregate (student numeric + school budget)
AGG_COLS = ['reading_score']
GROUP_COL = 'school_name'

Joined shape: (19585, 11)
   Student ID     student_name gender grade        school_name  reading_score  \
0           0     Paul Bradley      M   9th  Huang High School             66   
1           1     Victor Smith      M  12th  Huang High School             94   
2           2  Kevin Rodriguez      M  12th  Huang High School             90   

   math_score  School ID      type  size   budget  
0          79          0  District  2917  1910635  
1          61          0  District  2917  1910635  
2          60          0  District  2917  1910635  


In [97]:
# Compute all 5 aggregations per school
AGG_FUNCS = {'min': 'min', 'max': 'max', 'mean': 'mean', 'sum': 'sum', 'count': 'count'}

agg_results = {}
for name, func in AGG_FUNCS.items():
    agg_results[name] = joined.groupby(GROUP_COL)[AGG_COLS].agg(func)

# Sort aggregation functions by their mean output value across all AGG_COLS
# This gives a consistent axis order: lowest-output agg on left, highest on right
agg_mean_val = {
    name: agg_results[name].values.mean()
    for name in AGG_FUNCS.keys()
}
AGG_NAMES_SORTED = sorted(agg_mean_val, key=agg_mean_val.get)

print('Aggregation functions sorted by mean output value:')
for name in AGG_NAMES_SORTED:
    print(f'  {name:6s}: {agg_mean_val[name]:.2f}')

print('\nExample: count aggregation')
print(agg_results['count'])
print('\nExample: mean aggregation')
print(agg_results['mean'])

Aggregation functions sorted by mean output value:
  min   : 66.00
  mean  : 82.46
  max   : 99.00
  count : 2448.12
  sum   : 201015.75

Example: count aggregation
                       reading_score
school_name                         
Bailey High School              1714
Cabrera High School             1858
Figueroa High School            2949
Griffin High School             1468
Hernandez High School           4635
Huang High School               2917
Shelton High School             1761
Wilson High School              2283

Example: mean aggregation
                       reading_score
school_name                         
Bailey High School         80.858226
Cabrera High School        83.975780
Figueroa High School       81.158020
Griffin High School        83.816757
Hernandez High School      80.934412
Huang High School          81.182722
Shelton High School        83.725724
Wilson High School         83.989488


In [ ]:
# --------------------------------------------------------------------------
# Distance / similarity functions — all normalized to [0, 1]
#
# Histogram-based metrics (KL, JS, Hellinger, TV, Bhattacharyya):
#   Bin range is derived from ONLY the two arrays being compared.
#   This means: min vs mean uses their shared range (~50-100),
#   min vs sum uses their shared range (~50 to ~50000) → correctly shows max distance.
#
# Raw-value metrics (L1, L2, EMD): divided by shared value range → [0, 1].
# --------------------------------------------------------------------------

def to_prob_shared(a, b, bins=None):
    """
    Build histograms over the shared range of just these two arrays.
    Bin count defaults to Sturges' rule: ceil(log2(n) + 1) for small samples.
    """
    n = len(a)
    if bins is None:
        bins = int(np.ceil(np.log2(n) + 1))  # Sturges' rule
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if lo == hi:
        return np.array([1.0]), np.array([1.0])  # identical → zero distance
    pa, _ = np.histogram(a, bins=bins, range=(lo, hi), density=False)
    pb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=False)
    pa = pa.astype(float) + 1e-10
    pb = pb.astype(float) + 1e-10
    return pa / pa.sum(), pb / pb.sum()

def value_range(a, b):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    return (hi - lo) if (hi - lo) > 0 else 1.0

# Raw-value metrics (scale-aware)
def dist_l1(a, b, **_):
    return np.mean(np.abs(a - b)) / value_range(a, b)

def dist_l2(a, b, **_):
    return np.sqrt(np.mean((a - b) ** 2)) / value_range(a, b)

def dist_emd(a, b, **_):
    from scipy.stats import wasserstein_distance
    return wasserstein_distance(a, b) / value_range(a, b)

# Distribution-shape metrics (bin range = only the two arrays being compared)
def dist_kl(a, b, **_):
    from scipy.stats import entropy
    pa, pb = to_prob_shared(a, b)
    return 1 - np.exp(-entropy(pa, pb))

def dist_js(a, b, **_):
    from scipy.spatial.distance import jensenshannon
    pa, pb = to_prob_shared(a, b)
    # base=2 → JS divergence is in [0,1] in bits; sqrt gives JS distance in [0,1]
    return float(jensenshannon(pa, pb, base=2))

def dist_hellinger(a, b, **_):
    pa, pb = to_prob_shared(a, b)
    return np.sqrt(np.sum((np.sqrt(pa) - np.sqrt(pb)) ** 2)) / np.sqrt(2)

def dist_tv(a, b, **_):
    pa, pb = to_prob_shared(a, b)
    return 0.5 * np.sum(np.abs(pa - pb))

def dist_bhattacharyya(a, b, **_):
    pa, pb = to_prob_shared(a, b)
    bc = np.sum(np.sqrt(pa * pb))
    return 1 - np.exp(-(-np.log(bc + 1e-10)))

METRICS = {
    'L1':              dist_l1,
    'L2':              dist_l2,
    'KL Divergence':   dist_kl,
    'EMD':             dist_emd,
    'JS Distance':     dist_js,
    # 'Hellinger':       dist_hellinger,
    # 'Total Variation': dist_tv,
    # 'Bhattacharyya':   dist_bhattacharyya,
}

print('Metrics defined.')
print('  JS Distance uses log base 2 → divergence in [0,1] bits, distance (sqrt) in [0,1]')

In [99]:
# --------------------------------------------------------------------------
# Build 5×5 distance matrices, rows/cols ordered by mean aggregation output
# --------------------------------------------------------------------------

def build_matrix(metric_fn, agg_results, agg_cols, ordered_names):
    n = len(ordered_names)
    matrix = np.zeros((n, n))
    for i, name_i in enumerate(ordered_names):
        for j, name_j in enumerate(ordered_names):
            col_dists = []
            for col in agg_cols:
                a = agg_results[name_i][col].values
                b = agg_results[name_j][col].values
                col_dists.append(metric_fn(a, b))
            matrix[i, j] = np.mean(col_dists)
    return pd.DataFrame(matrix, index=ordered_names, columns=ordered_names)

matrices = {}
for metric_name, metric_fn in METRICS.items():
    matrices[metric_name] = build_matrix(metric_fn, agg_results, AGG_COLS, AGG_NAMES_SORTED)

print('Built matrices with sorted axis order:', AGG_NAMES_SORTED)
print('\nExample — KL Divergence matrix:')
matrices['KL Divergence'].round(4)

Built matrices with sorted axis order: ['min', 'mean', 'max', 'count', 'sum']

Example — KL Divergence matrix:


,min,mean,max,count,sum
min,0.0,1.0,1.0,1.0,1.0
mean,1.0,0.0,1.0,1.0,1.0
max,1.0,1.0,0.0,1.0,1.0
count,1.0,1.0,1.0,0.0,1.0
sum,1.0,1.0,1.0,1.0,0.0


In [ ]:
# --------------------------------------------------------------------------
# Visualize all 8 matrices as heatmaps
# Axis tick labels show the value range [min, max] across all AGG_COLS
# so it's clear why certain aggregations don't overlap in their distributions
# --------------------------------------------------------------------------

# Compute value range per aggregation across all AGG_COLS
agg_ranges = {
    name: (
        min(agg_results[name][col].values.min() for col in AGG_COLS),
        max(agg_results[name][col].values.max() for col in AGG_COLS)
    )
    for name in AGG_NAMES_SORTED
}

def fmt(v):
    """Format large numbers compactly (e.g. 201015 → 201K)."""
    if abs(v) >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if abs(v) >= 1_000:     return f'{v/1_000:.1f}K'
    return f'{v:.1f}'

tick_labels = [
    f'{n}\n[{fmt(agg_ranges[n][0])}, {fmt(agg_ranges[n][1])}]'
    for n in AGG_NAMES_SORTED
]

print('Aggregation value ranges across all columns:')
for name in AGG_NAMES_SORTED:
    lo, hi = agg_ranges[name]
    print(f'  {name:6s}: [{fmt(lo)}, {fmt(hi)}]')

fig, axes = plt.subplots(2, 4, figsize=(24, 11))
axes = axes.flatten()

for idx, (metric_name, mat) in enumerate(matrices.items()):
    ax = axes[idx]
    sns.heatmap(
        mat, annot=True, fmt='.3f', cmap='YlOrRd',
        ax=ax, linewidths=0.5, square=True,
        cbar_kws={'shrink': 0.8},
        xticklabels=tick_labels,
        yticklabels=tick_labels,
    )
    ax.set_title(metric_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Aggregation', fontsize=9)
    ax.set_ylabel('Correct Aggregation', fontsize=9)
    ax.tick_params(axis='both', labelsize=8)

plt.suptitle(
    'Aggregation Function Distance Matrices [0, 1]\n'
    'Axes sorted low→high by mean output value. Tick labels show [min, max] range across all columns.\n'
    f'Columns: {AGG_COLS}',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig('aggregation_distance_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: aggregation_distance_matrices.png')

In [101]:
# --------------------------------------------------------------------------
# Print all matrices for inspection
# --------------------------------------------------------------------------

for metric_name, mat in matrices.items():
    print(f'\n===== {metric_name} =====')
    print(mat.round(4).to_string())


===== L1 =====
          min    mean     max   count     sum
min    0.0000  0.7840  0.9167  0.5210  0.5358
mean   0.7840  0.0000  0.9120  0.5195  0.5358
max    0.9167  0.9120  0.0000  0.5179  0.5357
count  0.5210  0.5195  0.5179  0.0000  0.5314
sum    0.5358  0.5358  0.5357  0.5314  0.0000

===== L2 =====
          min    mean     max   count     sum
min    0.0000  0.7876  0.9204  0.5631  0.5742
mean   0.7876  0.0000  0.9154  0.5619  0.5742
max    0.9204  0.9154  0.0000  0.5607  0.5742
count  0.5631  0.5619  0.5607  0.0000  0.5695
sum    0.5742  0.5742  0.5742  0.5695  0.0000

===== KL Divergence =====
       min  mean  max  count  sum
min    0.0   1.0  1.0    1.0  1.0
mean   1.0   0.0  1.0    1.0  1.0
max    1.0   1.0  0.0    1.0  1.0
count  1.0   1.0  1.0    0.0  1.0
sum    1.0   1.0  1.0    1.0  0.0

===== EMD =====
          min    mean     max   count     sum
min    0.0000  0.7840  0.9167  0.5210  0.5358
mean   0.7840  0.0000  0.9120  0.5195  0.5358
max    0.9167  0.9120  0.0000 

In [102]:
# --------------------------------------------------------------------------
# Ranking analysis: for each metric, which wrong aggregation is closest
# to the correct one? (off-diagonal nearest neighbor)
# --------------------------------------------------------------------------

print('For each metric, given the correct agg = ROW, which predicted agg is LEAST wrong?\n')

for metric_name, mat in matrices.items():
    print(f'--- {metric_name} ---')
    for row in AGG_NAMES:
        # exclude diagonal (same agg)
        others = mat.loc[row].drop(row)
        closest = others.idxmin()
        farthest = others.idxmax()
        print(f'  correct={row:4s} | closest wrong={closest:4s} ({others[closest]:.4f}) | '
              f'farthest wrong={farthest:4s} ({others[farthest]:.4f})')
    print()

For each metric, given the correct agg = ROW, which predicted agg is LEAST wrong?

--- L1 ---
  correct=min  | closest wrong=count (0.5210) | farthest wrong=max  (0.9167)
  correct=max  | closest wrong=count (0.5179) | farthest wrong=min  (0.9167)
  correct=mean | closest wrong=count (0.5195) | farthest wrong=max  (0.9120)
  correct=sum  | closest wrong=count (0.5314) | farthest wrong=min  (0.5358)
  correct=count | closest wrong=max  (0.5179) | farthest wrong=sum  (0.5314)

--- L2 ---
  correct=min  | closest wrong=count (0.5631) | farthest wrong=max  (0.9204)
  correct=max  | closest wrong=count (0.5607) | farthest wrong=min  (0.9204)
  correct=mean | closest wrong=count (0.5619) | farthest wrong=max  (0.9154)
  correct=sum  | closest wrong=count (0.5695) | farthest wrong=min  (0.5742)
  correct=count | closest wrong=max  (0.5607) | farthest wrong=sum  (0.5695)

--- KL Divergence ---
  correct=min  | closest wrong=mean (1.0000) | farthest wrong=max  (1.0000)
  correct=max  | closest 